# Parte A — Aplicación RAG empresarial con NVIDIA NIM (Gradio)

> **Workshop RAG · SI3003 · Juan José Palacio**
> Tareas A1–A4: ejecutar la app, usar un **corpus propio**, mejorar el **prompt**
> y mostrar los **scores** de recuperación.

Esta versión es **autosuficiente**: configura NVIDIA NIM, carga los documentos,
crea un índice Qdrant temporal y publica una interfaz Gradio.

**Modelos:**
- Embeddings: `nvidia/nemotron-3-embed-1b`
- Generación: `nvidia/nemotron-3-super-120b-a12b`

**Qué cambié respecto al notebook base (resumen de tareas):**
| Tarea | Qué hice |
|---|---|
| **A1** | Ejecutar el pipeline completo end-to-end en Colab. |
| **A2** | Reemplacé el corpus demo por un **corpus propio** (plataforma *AuraCode*, 8 documentos en `data/company_documents.json`). |
| **A3** | Mejoré el **prompt** del sistema: reglas anti-alucinación, citado de fuente y formato de respuesta. |
| **A4** | La interfaz ahora muestra el **score de similitud** de cada fragmento recuperado. |

> En Colab, ejecuta las celdas en orden. La última celda mostrará una URL
> pública temporal de Gradio.


## 1. Instalar dependencias

Después de la instalación no suele ser necesario reiniciar el runtime. Si Colab lo solicita, reinícialo y vuelve a ejecutar desde la celda siguiente.


In [1]:
%pip install -qU gradio langchain langchain-core langchain-text-splitters langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client python-dotenv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.3/573.3 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 10.1 MB/s eta 0:00:00


## 2. Configurar la API key

En Colab, abre **🔑 Secrets**, crea un secreto llamado `NVIDIA_API_KEY` y habilita el acceso para este notebook. También puedes introducirla de forma oculta cuando la celda la solicite.


In [2]:
import os
import json
import getpass
from pathlib import Path
from typing import List, Dict

import gradio as gr
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

load_dotenv()

try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except (ImportError, KeyError):
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass.getpass("NVIDIA_API_KEY: ").strip()
if not NVIDIA_API_KEY:
    raise ValueError("Falta NVIDIA_API_KEY. Créala en https://build.nvidia.com/")

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
GENERATION_MODEL = "nvidia/nemotron-3-super-120b-a12b"
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"
COLLECTION = "company_docs_gradio"

print("✅ Configuración NVIDIA NIM lista")


✅ Configuración NVIDIA NIM lista


## 3. Cargar los documentos (A2 — corpus propio)

Esta app usa un **corpus propio** sobre *AuraCode*, una plataforma ficticia de
revisión de código asistida por IA. El archivo está en
`data/company_documents.json` con los campos `id`, `title`, `content` y
`metadata` (`department`, `type`).

En Colab, sube la carpeta `data/` (o solo `company_documents.json`) al entorno,
o apunta `DATA_PATH` a su ubicación. Si el archivo no existe, se usa un pequeño
conjunto de demostración como respaldo.


In [ ]:
DEMO_DOCUMENTS = [
    {
        "id": "producto_001", "title": "AuraCode — Planes y precios",
        "content": "El plan Team de AuraCode cuesta $39/mes por desarrollador con repositorios ilimitados. El plan Enterprise cuesta $69/mes por desarrollador con compromiso anual.",
        "metadata": {"department": "producto", "type": "pricing"},
    },
    {
        "id": "rrhh_001", "title": "Politica de trabajo remoto 2026",
        "content": "Los empleados pueden trabajar de forma remota hasta 4 dias por semana con aprobacion del lider de equipo. El dia presencial obligatorio es el martes.",
        "metadata": {"department": "rrhh", "type": "policy"},
    },
]


def load_documents(path: Path) -> List[Dict]:
    if not path.exists():
        print(f"i No se encontro {path}; se usaran documentos de demostracion.")
        return DEMO_DOCUMENTS

    with path.open(encoding="utf-8") as file:
        documents = json.load(file)

    required = {"id", "title", "content"}
    for index, document in enumerate(documents):
        missing = required - document.keys()
        if missing:
            raise ValueError(f"Al documento {index} le faltan campos: {missing}")
        document.setdefault("metadata", {})
    return documents


# A2 — corpus propio. En Colab sube la carpeta data/ al entorno.
DATA_PATH = Path("data/company_documents.json")
if not DATA_PATH.exists():
    DATA_PATH = Path("company_documents.json")  # respaldo si lo subiste suelto

company_documents = load_documents(DATA_PATH)
print(f"OK: {len(company_documents)} documentos disponibles (corpus propio AuraCode)")


## 4. Construir el pipeline RAG (A3 — prompt mejorado · A4 — scores)

Esta celda crea los fragmentos, calcula embeddings con NVIDIA NIM y los indexa
en Qdrant. El índice vive en memoria mientras la sesión de Colab esté activa.

**A3 — prompt mejorado.** El `system prompt` ahora: (1) prohíbe inventar y
obliga a decir "No tengo esa información en los documentos" cuando el contexto
no alcanza; (2) pide **citar el título** del documento que respalda la
respuesta; (3) fija el formato (español, conciso, sin relleno).

**A4 — scores.** Se expone una función `retrieve_with_scores()` que devuelve
cada fragmento junto con su **score de similitud coseno**, para mostrarlo en la
interfaz.


In [ ]:
def build_pipeline(documents: List[Dict]):
    embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, truncate="END")
    llm = ChatNVIDIA(
        model=GENERATION_MODEL,
        temperature=0.2,
        top_p=0.95,
        max_completion_tokens=700,
    )

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=400,
        chunk_overlap=40,
        separators=["\n\n", "\n", ".", " "],
    )

    chunks: List[Document] = []
    for document in documents:
        for index, text in enumerate(splitter.split_text(document["content"])):
            chunks.append(Document(
                page_content=text,
                metadata={
                    "source_id": document["id"],
                    "title": document["title"],
                    "chunk_index": index,
                    **document.get("metadata", {}),
                },
            ))

    embedding_dim = len(embeddings.embed_query("prueba de dimension"))
    qdrant_client = QdrantClient(":memory:")
    qdrant_client.create_collection(
        collection_name=COLLECTION,
        vectors_config=VectorParams(size=embedding_dim, distance=Distance.COSINE),
    )

    vectorstore = QdrantVectorStore(
        client=qdrant_client,
        collection_name=COLLECTION,
        embedding=embeddings,
    )
    vectorstore.add_documents(chunks)

    retriever = vectorstore.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": 3, "score_threshold": 0.3},
    )

    # A3 — prompt mejorado: anti-alucinacion + citar fuente + formato.
    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "Eres el asistente interno de AuraCode. Sigue estas reglas sin excepcion:\n"
            "1) Responde UNICAMENTE con informacion presente en el CONTEXTO.\n"
            "2) Si el contexto no contiene la respuesta, di exactamente: "
            "'No tengo esa informacion en los documentos.' No inventes datos.\n"
            "3) Cita entre parentesis el titulo del documento que respalda cada dato, "
            "por ejemplo: (Fuente: AuraCode - Planes y precios).\n"
            "4) Responde en espanol, de forma concisa y sin relleno.",
        ),
        ("human", "CONTEXTO:\n{context}\n\nPREGUNTA:\n{question}"),
    ])

    def format_docs(items: List[Document]) -> str:
        return "\n\n---\n\n".join(
            f"[{item.metadata.get('title', 'Sin titulo')}]\n{item.page_content}"
            for item in items
        )

    setup_and_retrieval = RunnableParallel(
        question=RunnablePassthrough(),
        documents=retriever,
    )
    answer_from_documents = (
        {
            "context": lambda item: format_docs(item["documents"]),
            "question": lambda item: item["question"],
        }
        | prompt
        | llm
        | StrOutputParser()
    )
    pipeline = setup_and_retrieval.assign(answer=answer_from_documents)

    # A4 — helper para obtener los scores de similitud de cada fragmento.
    def retrieve_with_scores(question: str, k: int = 3):
        return vectorstore.similarity_search_with_score(question, k=k)

    return pipeline, chunks, retrieve_with_scores


print("Construyendo el indice...")
rag_pipeline, indexed_chunks, retrieve_with_scores = build_pipeline(company_documents)
titles = [document["title"] for document in company_documents]
print(f"OK: pipeline listo: {len(titles)} documentos, {len(indexed_chunks)} chunks")


## 5. Crear la interfaz Gradio

In [ ]:
def chat(question: str, history: list | None):
    history = history or []
    question = (question or "").strip()
    if not question:
        return "", history, "Escribe una pregunta."

    try:
        result = rag_pipeline.invoke(question)
        answer = result["answer"].strip()

        # A4 — recuperar fragmentos CON su score de similitud.
        scored = retrieve_with_scores(question, k=3)
        if scored:
            lines = ["**Fuentes recuperadas (con score de similitud):**"]
            seen = set()
            for doc, score in scored:
                title = doc.metadata.get("title", "Sin titulo")
                dept = doc.metadata.get("department", "-")
                key = (title, round(float(score), 3))
                if key in seen:
                    continue
                seen.add(key)
                lines.append(f"- **{title}** · _{dept}_ · score = `{float(score):.3f}`")
            sources_md = "\n".join(lines)
        else:
            sources_md = "No se encontraron documentos relevantes."

    except Exception as error:
        answer = "Ocurrio un error al consultar el pipeline. Revisa el detalle debajo."
        sources_md = f"**Error:** `{type(error).__name__}: {error}`"

    history = history + [
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ]
    return "", history, sources_md


def clear_chat():
    return [], ""


with gr.Blocks(title="RAG empresarial AuraCode con NVIDIA NIM") as demo:
    gr.Markdown(
        "# RAG empresarial — AuraCode\n"
        "Consulta documentos internos mediante **LangChain + NVIDIA NIM + Qdrant**. "
        "Cada respuesta muestra las **fuentes recuperadas con su score**."
    )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Documentos indexados")
            gr.Markdown("\n".join(f"- {title}" for title in titles))
            gr.Markdown(f"*{len(indexed_chunks)} chunks en el indice*")
            gr.Markdown("---")
            gr.Markdown(
                f"**Embeddings:** `{EMBEDDING_MODEL}`  \n"
                f"**LLM:** `{GENERATION_MODEL}`  \n"
                "**Vector store:** Qdrant en memoria  \n"
                "**Retrieval:** k=3, threshold=0.3"
            )

        with gr.Column(scale=2):
            chatbot = gr.Chatbot(height=430, label="Chat")
            with gr.Row():
                question_box = gr.Textbox(
                    placeholder="Haz una pregunta sobre los documentos de AuraCode...",
                    label="Pregunta",
                    lines=2,
                    scale=4,
                )
                send_button = gr.Button("Enviar", variant="primary", scale=1)

            sources_box = gr.Markdown()
            clear_button = gr.Button("Limpiar", variant="secondary")

    gr.Examples(
        examples=[
            ["Cuanto cuesta el plan Enterprise de AuraCode?"],
            ["Cuantos dias se puede trabajar de forma remota?"],
            ["Que pasa con los reembolsos mayores a 150 dolares?"],
            ["Que lenguajes de programacion soporta AuraCode?"],
            ["Cual es el salario del CEO?"],
        ],
        inputs=question_box,
        label="Preguntas de ejemplo (la ultima NO esta en el corpus: debe reconocer que no sabe)",
    )

    send_button.click(
        chat,
        inputs=[question_box, chatbot],
        outputs=[question_box, chatbot, sources_box],
    )
    question_box.submit(
        chat,
        inputs=[question_box, chatbot],
        outputs=[question_box, chatbot, sources_box],
    )
    clear_button.click(clear_chat, outputs=[chatbot, sources_box])


## 6. Ejecutar la aplicación en Colab

`share=True` crea el enlace público temporal. La aplicación estará disponible mientras esta celda y el runtime de Colab continúen activos.


In [9]:
demo.queue().launch(
    share=True,
    debug=True,
    show_error=True,
)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9745d08ed77e63aba6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://9745d08ed77e63aba6.gradio.live


### Detener o volver a ejecutar

- Interrumpe la última celda para detener el servidor.
- Antes de volver a ejecutarla, puedes cerrar instancias previas con `gr.close_all()`.
- El enlace público cambia cada vez que se inicia la aplicación.


In [12]:
# Ejecuta esta celda solo cuando quieras cerrar servidores Gradio anteriores.
gr.close_all()
